In [63]:
from malmo import MalmoPython
import os
import sys

# Initialize the AgentHost
agent_host = MalmoPython.AgentHost()

# Pass an empty list [] so it doesn't try to parse Jupyter's internal flags
try:
    agent_host.parse([]) 
except RuntimeError as e:
    print('ERROR:', e)

print("Agent Host is now defined and ready.")

Agent Host is now defined and ready.


In [64]:
import malmo.run_mission as run_mission
from malmo import MalmoPython
import os

# Define the path to your file inside the Docker environment
xml_file_path = '/home/malmo/Documents/work/parkour_simple_jump.xml'

# Read the file
with open(xml_file_path, 'r') as f:
    parkour_xml = f.read()

# Initialize the mission
agent_host = MalmoPython.AgentHost()
my_mission = MalmoPython.MissionSpec(parkour_xml, True)
my_mission_record = MalmoPython.MissionRecordSpec()

print("XML File successfully loaded from: {0}".format(xml_file_path))

XML File successfully loaded from: /home/malmo/Documents/work/parkour_simple_jump.xml


In [65]:
import time

# 1. Initialize the Mission Record (used for saving videos/data of the run)
my_mission_record = MalmoPython.MissionRecordSpec()

# 2. Attempt to start the mission
try:
    # This sends the XML to the Minecraft client
    agent_host.startMission(my_mission, my_mission_record)
except RuntimeError as e:
    print("Error starting mission: {0}".format(e))

# 3. Wait for the mission to actually start
# Minecraft takes a few seconds to build the world and spawn the agent
print("Waiting for the mission to start", end=' ')
world_state = agent_host.getWorldState()

while not world_state.has_mission_begun:
    print(".", end="")
    time.sleep(0.1)
    world_state = agent_host.getWorldState()
    
    # Check if there are errors during startup
    for error in world_state.errors:
        print("Error: {0}".format(error.text))

print("\nMission is now running! Check your VNC window.")

Waiting for the mission to start ...................................
Mission is now running! Check your VNC window.


In [66]:
# Jump Test

In [49]:
import json

# 1. Start moving forward
agent_host.sendCommand("move 1")
print("Moving toward the gap...")

# 2. Timing the jump 
# Based on the spawn point (0.5, 46, 0.5), the gap is at Z=1.
# We'll wait a fraction of a second to get some speed.
time.sleep(0.4) 

# 3. Trigger the Jump
agent_host.sendCommand("jump 1")
print("Jumping!")

# 4. Air-time
# Wait while the agent is in the air
time.sleep(0.5)

# 5. Land and stop jumping
agent_host.sendCommand("jump 0")

# 6. Check if we survived
time.sleep(1) # Wait to settle
world_state = agent_host.getWorldState()

if world_state.number_of_observations_since_last_state > 0:
    msg = world_state.observations[-1].text
    data = json.loads(msg)
    current_y = data.get('YPos', 0)
    current_z = data.get('ZPos', 0)
    
    if current_y < 44:
        print("Outcome: FELL INTO THE VOID at Z={0:.2f}".format(current_z))
    else:
        print("Outcome: LANDED SAFELY at Z={0:.2f}!".format(current_z))

# Stop moving regardless of outcome
agent_host.sendCommand("move 0")

Moving toward the gap...
Jumping!
Outcome: LANDED SAFELY at Z=0.50!


In [39]:
import json
import time

print("Tracking Agent Position... (Press 'Interrupt' in the toolbar to stop)")

try:
    while True:
        # 1. Get the world state from the agent host
        world_state = agent_host.getWorldState()

        # 2. Check if there is new observation data
        if world_state.number_of_observations_since_last_state > 0:
            # The last observation is the most recent one
            msg = world_state.observations[-1].text
            data = json.loads(msg)
            
            # 3. Extract X, Y, and Z
            # .get() is safer in case a specific coordinate is missing
            x = data.get('XPos', 0.0)
            y = data.get('YPos', 0.0)
            z = data.get('ZPos', 0.0)
            
            # 4. Print formatted string (Python 3.5 style)
            # \r (return) and end='' keep the output on a single line
            print("\rPosition -> X: {0:.2f} | Y: {1:.2f} | Z: {2:.2f}".format(x, y, z), end='')
        
        time.sleep(0.1) 
        
        # Stop tracking if the mission ends
        if not world_state.is_mission_running:
            print("\nMission finished.")
            break
            
except KeyboardInterrupt:
    print("\nTracking stopped by user.")

Tracking Agent Position... (Press 'Interrupt' in the toolbar to stop)
Position -> X: 0.50 | Y: 46.00 | Z: 0.50
Mission finished.
